# 🌸 Task 1 · Iris Flower Classification

**Objective:** Train a machine learning classification model to identify the species of an iris flower (*Setosa*, *Versicolor*, or *Virginica*) from its physical measurements.

**Tech stack:** Python, scikit-learn, pandas, matplotlib, seaborn, Jupyter Notebook

**Notebook outline**
1. Load the dataset
2. Exploratory Data Analysis (EDA)
3. Visualisations
4. Feature selection discussion
5. Train/test split (80/20)
6. Train classifiers
7. Evaluate each model
8. Compare models and pick the best
9. Conclusion

## 0. Imports
All libraries used in the notebook are imported here in one place.

In [ ]:
# Data handling
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset, splitting and cross-validation
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Preprocessing and pipelines (scaling helps Logistic Regression and KNN)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Evaluation metrics
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

# Make plots look nicer and results easy to reproduce
sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42

## 1. Load the Dataset
The Iris dataset is built into scikit-learn, so **no download is needed**. It has 150 flowers (50 per species) with 4 measurements each, all in centimetres.

In [ ]:
# Load the dataset (returns a Bunch object with data, target, names, etc.)
iris = load_iris()

# Convert the features into a pandas DataFrame with readable column names
df = pd.DataFrame(iris.data, columns=iris.feature_names)

# Add the numeric target (0, 1, 2) and a readable species name column
df["target"] = iris.target
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))

# Look at the first few rows
df.head()

## 2. Exploratory Data Analysis (EDA)
Before modelling, we check the size of the data, the data types, missing values, class balance and basic statistics.

In [ ]:
# Shape: (rows, columns)
print("Shape of the dataset:", df.shape)

# Data types and non-null counts
print("\nData types / info:")
df.info()

In [ ]:
# Null value check: count of missing values in each column
print("Missing values per column:")
print(df.isnull().sum())

# Check for duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Class distribution: how many flowers of each species?
print("Class distribution:")
print(df["species"].value_counts())

In [ ]:
# Descriptive statistics for the 4 measurement columns
df[iris.feature_names].describe().round(2)

In [ ]:
# Mean of each feature per species - a quick way to spot differences
df.groupby("species")[iris.feature_names].mean().round(2)

**Interpretation**
- The dataset has **150 rows** and **4 numeric features** plus the species label.
- There are **no missing values**, so no cleaning or imputation is needed.
- The classes are **perfectly balanced** (50 each), so accuracy is a fair metric here.
- The per-species means already show that **petal length and petal width** differ a lot between species (Setosa has very small petals), while sepal measurements overlap more.

## 3. Visualisations

### 3.1 Pairplot
The pairplot shows every feature against every other feature, coloured by species. The diagonal shows each feature's distribution.

In [ ]:
# Pairplot coloured by species (drop the numeric 'target' column first)
sns.pairplot(df.drop(columns="target"), hue="species", diag_kind="kde", height=2.3)
plt.suptitle("Iris Pairplot by Species", y=1.02, fontsize=14)
plt.show()

**Interpretation:** *Setosa* (one colour) is completely separated from the other two species, especially in the petal plots. *Versicolor* and *Virginica* overlap a little, but they are still mostly separable using petal length and petal width.

### 3.2 Box plots for each feature
Box plots show the median, spread and outliers of each feature for every species.

In [ ]:
# One box plot per feature, grouped by species
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

for ax, feature in zip(axes.ravel(), iris.feature_names):
    sns.boxplot(data=df, x="species", y=feature, hue="species", ax=ax, legend=False)
    ax.set_title(feature)
    ax.set_xlabel("")

plt.suptitle("Feature Distributions by Species", fontsize=14)
plt.tight_layout()
plt.show()

**Interpretation:** For petal length and petal width, the boxes of the three species barely overlap, so these features separate the classes very well. Sepal width is the weakest: the boxes overlap heavily. A few mild outliers appear (for example in sepal width), but there are too few to matter.

### 3.3 Correlation heatmap
Shows how strongly the features move together (1 = perfectly positive, -1 = perfectly negative).

In [ ]:
# Correlation matrix between the 4 numeric features
corr = df[iris.feature_names].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

**Interpretation:** Petal length and petal width are very strongly correlated (about 0.96), and both are also strongly correlated with sepal length. This means the petal features carry similar information.

## 4. Feature Selection Discussion
**Which features are the most discriminative?**

We use two kinds of evidence: (a) the plots above and (b) a quantitative measure, the feature importances from a Random Forest.

In [ ]:
# Fit a Random Forest on the full data just to measure feature importance
rf_importance = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf_importance.fit(df[iris.feature_names], df["target"])

# Put the importances in a sorted table
importances = (pd.Series(rf_importance.feature_importances_, index=iris.feature_names)
                 .sort_values(ascending=False))
print(importances.round(3))

# Bar chart of the importances
plt.figure(figsize=(7, 4))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, legend=False)
plt.xlabel("Importance")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

**Discussion**
- **Petal length and petal width are the most discriminative features.** Together they usually account for the large majority of the importance score. In the plots, they separate Setosa perfectly and Versicolor vs. Virginica quite well.
- **Sepal length** helps a little, mainly to separate Versicolor from Virginica.
- **Sepal width** is the least useful: the species overlap strongly on it.
- Because petal length and width are highly correlated, they are partly redundant. Importance is shared between them, so neither alone tells the full story.

**Decision:** The dataset only has 4 features and all of them are cheap to use, so we keep **all 4 features** for the models. (Using only the two petal features would also work well, which is a nice experiment to try later.)

## 5. Train/Test Split (80/20)
We keep 20% of the data hidden from the models so we can test how they perform on **unseen** flowers.
- `stratify=y` keeps the same species proportions in both sets.
- `random_state=42` makes the split reproducible.

In [ ]:
# Features (X) and target labels (y)
X = df[iris.feature_names]
y = df["target"]

# 80% training, 20% testing, stratified by species
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training set:", X_train.shape)
print("Test set    :", X_test.shape)
print("\nSpecies counts in the test set:")
print(y_test.map(dict(enumerate(iris.target_names))).value_counts())

## 6. Train the Classifiers
We train **four** different classifiers:

| Model | Idea in one line |
|---|---|
| Logistic Regression | Learns linear boundaries between classes |
| K-Nearest Neighbours (KNN) | Looks at the 5 closest training flowers and votes |
| Decision Tree | Learns a series of yes/no questions on the features |
| Random Forest | Many decision trees voting together |

Logistic Regression and KNN depend on feature scale, so they are wrapped in a **pipeline** with `StandardScaler`. Tree-based models do not need scaling.

In [ ]:
# Dictionary of models so we can loop over them easily
models = {
    "Logistic Regression": make_pipeline(StandardScaler(),
                                         LogisticRegression(max_iter=200, random_state=RANDOM_STATE)),
    "K-Nearest Neighbours": make_pipeline(StandardScaler(),
                                          KNeighborsClassifier(n_neighbors=5)),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

# Train every model on the training data
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Trained: {name}")

## 7. Evaluate Each Model
For every model we report:
- **Accuracy** - fraction of test flowers classified correctly
- **Confusion matrix** - which species get mixed up (rows = true, columns = predicted)
- **Classification report** - precision, recall and F1 for each species

In [ ]:
# Store results so we can compare the models later
results = []
class_names = iris.target_names

for name, model in models.items():
    # Predict on the unseen test data
    y_pred = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    results.append({"Model": name, "Test Accuracy": acc, "Macro F1": f1})

    # Text output
    print("=" * 60)
    print(f"{name}  |  Accuracy: {acc:.3f}")
    print("=" * 60)
    print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

    # Confusion matrix as a heatmap
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4.5, 3.8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, cbar=False)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {name}")
    plt.tight_layout()
    plt.show()

## 8. Compare Models and Choose the Best
A single 30-flower test set is small, so one lucky or unlucky split can change the scores by a few percent. To make the comparison more reliable, we also run **5-fold stratified cross-validation** on the whole dataset.

In [ ]:
# Summary table of test-set results
results_df = pd.DataFrame(results)

# 5-fold stratified cross-validation on the full dataset for each model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_mean, cv_std = [], []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
    cv_mean.append(scores.mean())
    cv_std.append(scores.std())

results_df["CV Mean Accuracy"] = cv_mean
results_df["CV Std"] = cv_std

# Sort by cross-validated accuracy, then by test accuracy
results_df = results_df.sort_values(["CV Mean Accuracy", "Test Accuracy"], ascending=False).reset_index(drop=True)
results_df.round(3)

In [ ]:
# Visual comparison of the models
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(results_df))
width = 0.38

ax.bar(x - width/2, results_df["Test Accuracy"], width, label="Test accuracy")
ax.bar(x + width/2, results_df["CV Mean Accuracy"], width,
       yerr=results_df["CV Std"], capsize=4, label="5-fold CV accuracy")

ax.set_xticks(x)
ax.set_xticklabels(results_df["Model"], rotation=15)
ax.set_ylim(0.8, 1.02)
ax.set_ylabel("Accuracy")
ax.set_title("Model Comparison")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Pick the best model automatically from the table (top row after sorting)
best_name = results_df.loc[0, "Model"]
print("Best-performing model:", best_name)
print(results_df.round(3).loc[0].to_string())

### Best model and justification

All four models do very well on Iris (typically **90-100% accuracy**), because the species are well separated. The table and chart above show the ranking for this run; the justification below holds for the model at the top of the table.

**How to choose between them:**
1. **Accuracy and F1** - Pick the model with the highest test accuracy and macro F1.
2. **Consistency** - Cross-validation accuracy with a low standard deviation means the model does well on *different* splits, not just one lucky one. This matters more than a 1-flower difference on the test set.
3. **Simplicity and interpretability** - If two models are tied, prefer the simpler one. Logistic Regression and a shallow Decision Tree are easy to explain; Random Forest is more of a black box.
4. **Where the errors are** - In every confusion matrix, Setosa is never confused with anything. Any mistakes are between **Versicolor and Virginica**, which overlap slightly.

**Final declaration:** the model in the top row of the comparison table (printed above as *Best-performing model*) is declared the best, as it combines the highest cross-validated accuracy with strong test accuracy and F1. If several models are tied within about 1%, Logistic Regression is a very good practical choice thanks to its simplicity and interpretability.

## 9. Conclusion
- The Iris dataset is **clean, balanced and small** (150 samples, 4 features, no missing values).
- **Petal length and petal width** are the most discriminative features; sepal width is the weakest.
- **Setosa** is perfectly separable; the only real confusion is between **Versicolor and Virginica**.
- Four classifiers (Logistic Regression, KNN, Decision Tree, Random Forest) all reach high accuracy, and the best one was selected using test metrics plus 5-fold cross-validation.
- **Limitations:** the dataset is tiny and very easy, so these scores will not transfer to harder, noisier real-world problems.
- **Next steps:** try hyperparameter tuning (`GridSearchCV`), try other models such as SVM, and train on only the two petal features to see if accuracy stays the same.